In [1]:
from pathlib import Path
import re
import os
import dotenv
import pymupdf4llm
import pymupdf
import asyncio
import traceback
import pandas as pd
from tabulate import tabulate

from langchain_text_splitters import RecursiveCharacterTextSplitter

from llms_and_models import OpenAIModel
from chunks import TableChunk
from postgres import save_document_chunks, insert_pdf
from qdrant import upload_to_qdrant, format_embeddings


c:\Users\UserAdmin\Documents\Multimodal-LLM\.llm-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'llms_and_models'

In [ ]:
def same_table(prev_page,prev_table,curr_page,curr_table,tolerance=5):

    if prev_table.col_count != curr_table.col_count:
        return False

    prev_bbox = prev_table.bbox
    curr_bbox = curr_table.bbox

    if abs(prev_bbox[0]-curr_bbox[0]) > tolerance or abs(prev_bbox[2]-curr_bbox[2]) > tolerance:
        return False

    return True

In [ ]:
def get_table_caption(page,table):
    
    top_text = ""
    bottom_text = ""

    table_top = table.bbox[1]
    table_bottom = table.bbox[3]
    closest_block_top = float('inf')
    closest_block_bottom = float('inf')

    blocks = page.get_text('blocks',sort=True)

    for block in blocks:
        block_text = block[4]
        block_top = block[1]
        block_bottom = block[3]
        dist_top = table_top-block_bottom
        dist_bottom = block_top-table_bottom
        if 0 < dist_top < closest_block_top:
            top_text = block_text
            closest_block_top = dist_top
        if 0 < dist_bottom < closest_block_bottom:
            bottom_text = block_text
            closest_block_bottom = dist_bottom

    table_texts = top_text + bottom_text

    return table_texts

In [ ]:
async def extract_tables(filepath):

    all_tables = []
    last_seen_table = -1

    with pymupdf.open(filepath) as doc:

        for page_no, page in enumerate(doc):

            found_tables = page.find_tables()

            for i,table in enumerate(found_tables.tables):

                item = {
                        'table_texts' : [],
                        'tables' : [],
                        'pages' : []
                    }
                
                table_texts = get_table_caption(page,table)
                is_possible_continuation = (page_no - last_seen_table == 1 and i == 0)

                if not is_possible_continuation:
                    #Not possible for tables to be continutation of the previous page
                    item['table_texts'].append(table_texts)
                    item['tables'].append(table)
                    item['pages'].append(page_no)
                    all_tables.append(item)
                    last_seen_table = page_no

                else:
                    #Possible for tables to be continutations of each other
                    prev_table_item = all_tables[-1]
                    most_recent_table = prev_table_item['tables'][-1]
                    is_same_table = same_table(last_seen_table, most_recent_table, page_no, table)

                    if is_same_table:
                        #The previous table and the current one is deemed to be the same table split amongst pages
                        prev_table_item['table_texts'].append(table_texts)
                        prev_table_item['tables'].append(table)
                        prev_table_item['pages'].append(page_no)
                        last_seen_table = page_no

                    else:
                        #Table on previous page is distinct from table on current page
                        item['table_texts'].append(table_texts)
                        item['tables'].append(table)
                        item['pages'].append(page_no)
                        all_tables.append(item)
                        last_seen_table = page_no

    return all_tables

In [ ]:
async def format_chunks(filepath,all_tables):

    with pymupdf.open(filepath) as doc:
        full_text = pymupdf4llm.to_markdown(doc)

    model = OpenAIModel()
    doc_name = filepath.name
    table_chunks = []

    for item in all_tables:
        #One item is one table
        table_chunk = TableChunk()

        combined_text = "\n".join(item['table_texts'])
        combined_table = []
        headers = []

        for i,table in enumerate(item['tables']):
            if i == 0:
                #First joining table need to extract out headers
                df = table.to_pandas()
                df.columns = [str(col).strip() for col in df.columns]
                headers = df.columns.tolist()
                combined_table.append(df)
            else:
                #Other tables in the sequence need to add proper headers to data for proper concat behaviour
                rows = table.extract()
                if rows:
                    first_row = [str(header).strip() for header in rows[0]]
                    if first_row == headers:
                        #Headers are present, but we can format them nicely alltogether
                        data = rows[1:]
                    else:
                        #Headers are not present, need to add them on later
                        data = rows
                    
                    if data and len(data[0]) == len(headers):
                        df = pd.DataFrame(data,columns=headers)
                        combined_table.append(df)
                    elif data:
                        df = pd.DataFrame(data,columns=headers[:len(df.columns)]) 
                        combined_dfs.append(df)

        combined_table = pd.concat(combined_table, axis=0, ignore_index=True)

        combined_table.head()
        
        combined_table.to_html('table-test.html',index=False)

        # contents = f"""
        # Texts around tables : {combined_text}
        # Tables : 
        # {combined_table}
        # """
        # table_context = await model.get_context(full_text, contents)

        # table_chunk.document_name = doc_name
        # table_chunk.context = table_context
        # table_chunk.content['table'] = combined_table
        # table_chunk.content['surrounding_texts'] = combined_text
        # table_chunk.metadata = {'pages' : item['pages']}

        # table_chunks.append(table_chunk)

    return table_chunks

In [ ]:
async def process_tables(folder_path):

    try:
        if folder_path.is_dir():
            for file in folder_path.iterdir():

                chunks = await extract_tables(file)

                print(f'\tFinished extracting tables\n\n')

                returned_chunks = save_document_chunks(file.name,chunks)

                print(f'\tFinished saving chunks into postgresdb\n\n')

                embeddings = await embed_chunks(returned_chunks)

                print(f'\tFinished getting embeddings\n\n')

                upload_to_qdrant(embeddings)

                print(f'\tFinished uploading chunk embeddings to qdrant\n\n')

                print(f'Finished processing\n\n')

            print(f'\nFinished processing all files\n')

    except Exception as e:
        print(f'Unable to process tables, error {e}')
        traceback.print_exc() 
        raise

In [2]:

if __name__ == "__main__":

    async def main():

        test_pdf_path = Path(r'C:\Users\UserAdmin\Documents\Multimodal-LLM\pdfs\tables\government-data-security-policies.pdf')
        results_path = Path(os.getenv('test_results_path'))
        file = results_path / "tables-test.txt"

        all_table_chunks = await extract_tables(test_pdf_path)
        clean_chunks = await format_chunks(test_pdf_path, all_table_chunks)

        # with open(file, "w", encoding="utf-8") as f:
        #     for i, chunk in enumerate(clean_chunks):
        #         f.write(f"Chunk number {i}\n")
        #         f.write(f"Chunk from {chunk.document_name}\n")
        #         f.write(f"Chunk type : {chunk.type}\n")
        #         f.write(f"Chunk context : {chunk.context}\n")
        #         f.write(f"Chunk content : {chunk.content}\n")
        #         f.write(f"Chunk metadata : {chunk.metadata}\n")
        #         f.write(f'\n\n')

    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop